# Colab Training Notebook

This notebook is the complete Colab workflow for training the Mistral + QLoRA model, generating evaluation outputs, and saving artifacts to Google Drive.

## Recommended setup
- Use a GPU runtime.
- Clone the GitHub repo inside Colab.
- Mount Google Drive if you want checkpoints and outputs to persist after the session ends.


## 1. Create and Configure a Google Colab Notebook

In Colab, change the runtime type to **GPU** first, then run the environment check below.


In [13]:
import os
import sys
import platform

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Install and Import Dependencies

Install the project packages and common plotting libraries. Keep the versions pinned for reproducibility.


In [14]:
!pip -q install transformers==4.46.3 datasets==2.19.0 peft==0.18.1 trl==0.8.6 bitsandbytes accelerate==1.13.0 sentencepiece pandas matplotlib seaborn

import json
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('Dependencies installed.')


Dependencies installed.


## 3. Clone the GitHub Repo

Clone the project inside Colab and run everything from the repository root.


In [15]:
!git clone https://github.com/DEATHGATE01/llm-finetune.git
%cd llm-finetune

PROJECT_ROOT = Path('/content/llm-finetune')
DATA_PATH = PROJECT_ROOT / 'data' / 'final_dataset.json'
RESULTS_DIR = PROJECT_ROOT / 'results'

print('Project root:', PROJECT_ROOT)
print('Dataset exists:', DATA_PATH.exists())


Cloning into 'llm-finetune'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 53 (delta 10), reused 53 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 86.93 KiB | 6.21 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/llm-finetune/llm-finetune/llm-finetune/llm-finetune
Project root: /content/llm-finetune
Dataset exists: True


## 3b. Optional: Mount Google Drive for Persistent Saves

Use this only if you want checkpoints and logs to stay after Colab disconnects.


In [16]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/llm-finetune')
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'
DRIVE_LOGS_DIR = DRIVE_RESULTS_DIR / 'logs'
DRIVE_MODEL_DIR = DRIVE_RESULTS_DIR / 'finetuned_model'

DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_LOGS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

os.environ['TRAIN_OUTPUT_DIR'] = str(DRIVE_MODEL_DIR)
os.environ['TRAIN_LOGGING_DIR'] = str(DRIVE_LOGS_DIR)

print('Drive root:', DRIVE_ROOT)
print('Training output dir:', os.environ['TRAIN_OUTPUT_DIR'])
print('Training log dir:', os.environ['TRAIN_LOGGING_DIR'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/llm-finetune
Training output dir: /content/drive/MyDrive/llm-finetune/results/finetuned_model
Training log dir: /content/drive/MyDrive/llm-finetune/results/logs


## 4. Configure Mistral + QLoRA Training

Set the environment variables for the real training run and validate that the dataset exists in the cloned repository.


In [17]:
os.environ['USE_4BIT'] = 'true'
os.environ['MODEL_NAME'] = 'mistralai/Mistral-7B-Instruct-v0.2'
os.environ['MAX_LENGTH'] = '1024'
os.environ['TRAIN_FILE'] = str(PROJECT_ROOT / 'data' / 'processed' / 'train_instruct.jsonl')
os.environ['VAL_FILE'] = str(PROJECT_ROOT / 'data' / 'processed' / 'val_instruct.jsonl')

print('USE_4BIT =', os.environ['USE_4BIT'])
print('MODEL_NAME =', os.environ['MODEL_NAME'])
print('MAX_LENGTH =', os.environ['MAX_LENGTH'])
print('TRAIN_FILE =', os.environ['TRAIN_FILE'])
print('VAL_FILE =', os.environ['VAL_FILE'])
print('Training dataset exists:', Path(os.environ['TRAIN_FILE']).exists())
print('Validation dataset exists:', Path(os.environ['VAL_FILE']).exists())

USE_4BIT = true
MODEL_NAME = mistralai/Mistral-7B-Instruct-v0.2
MAX_LENGTH = 1024
TRAIN_FILE = /content/llm-finetune/data/processed/train_instruct.jsonl
VAL_FILE = /content/llm-finetune/data/processed/val_instruct.jsonl
Training dataset exists: True
Validation dataset exists: True


## 5. Run Training

Run the real training job from the repository root. If you want a quick proof of life first, reduce the sample counts.


In [18]:
!python -m src.training.train --max-train-samples 100 --max-eval-samples 32


2026-04-05 23:04:00.254336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775430240.451404   17732 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775430240.508470   17732 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775430240.910693   17732 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775430240.910746   17732 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775430240.910751   17732 computation_placer.cc:177] computation placer alr

## 6. Generate Evaluation Outputs

After training, create base and fine-tuned predictions, then compare their metrics.


In [19]:
FINETUNED_MODEL_PATH = Path(os.environ.get('TRAIN_OUTPUT_DIR', str(RESULTS_DIR / 'finetuned_model')))
BASE_OUTPUT = RESULTS_DIR / 'base_outputs' / 'predictions.json'
FINETUNED_OUTPUT = RESULTS_DIR / 'finetuned_outputs' / 'predictions.json'
METRICS_OUTPUT = RESULTS_DIR / 'metrics' / 'comparison_report.json'

!python -m src.evaluation.inference --output {BASE_OUTPUT} --max-samples 25
!python -m src.evaluation.inference --model-name {FINETUNED_MODEL_PATH} --output {FINETUNED_OUTPUT} --max-samples 25
!python -m src.evaluation.compare --base-file {BASE_OUTPUT} --finetuned-file {FINETUNED_OUTPUT} --report-file {METRICS_OUTPUT}

print('Base outputs:', BASE_OUTPUT)
print('Fine-tuned outputs:', FINETUNED_OUTPUT)
print('Metrics report:', METRICS_OUTPUT)

2026-04-05 23:18:03.247353: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775431083.276490   21446 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775431083.285707   21446 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775431083.317834   21446 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775431083.317880   21446 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775431083.317885   21446 computation_placer.cc:177] computation placer alr

## 7. Save Outputs and Export Notebook

Copy the final artifacts into Google Drive so they remain available after the Colab runtime ends.


In [20]:
import shutil

if 'DRIVE_RESULTS_DIR' in globals():
    for name in ['finetuned_model', 'logs', 'base_outputs', 'finetuned_outputs', 'metrics']:
        source_dir = RESULTS_DIR / name
        target_dir = DRIVE_RESULTS_DIR / name
        if source_dir.exists():
            if target_dir.exists():
                shutil.rmtree(target_dir)
            shutil.copytree(source_dir, target_dir)
            print(f'Copied {source_dir} -> {target_dir}')
else:
    print('Drive is not mounted. If you want persistence, mount Drive before training.')


Copied /content/llm-finetune/results/base_outputs -> /content/drive/MyDrive/llm-finetune/results/base_outputs
Copied /content/llm-finetune/results/finetuned_outputs -> /content/drive/MyDrive/llm-finetune/results/finetuned_outputs
Copied /content/llm-finetune/results/metrics -> /content/drive/MyDrive/llm-finetune/results/metrics


## 8. What to Expect

- The training cell saves the fine-tuned checkpoint to the output directory.
- The evaluation cell creates prediction files and a metrics report.
